# lstm-forecast quickstart

End-to-end: load data (synthetic fallback offline), forecast with conformal intervals, benchmark against baselines, and generate an AI insight.

> Forecasts are uncertain and **not financial advice**.

In [ ]:
from lstm_forecast import Forecaster, Pipeline
from lstm_forecast.data import load_prices, add_finance_features
from lstm_forecast.transforms import default_finance_transformer

df = load_prices('AAPL', allow_synthetic_fallback=True)
feat = add_finance_features(df, fourier_periods=(5.0,))
df.tail()

In [ ]:
f = Forecaster(
    y=feat['close'], current_dates=feat.index,
    future_dates=21, test_length=42, exog=feat.drop(columns=['close']),
)
transformer, reverter = default_finance_transformer(seasonal_period=5)
result = Pipeline(transformer=transformer, reverter=reverter).fit_predict(
    f, lags=21, epochs=60, alpha=0.1,
)
result.metrics_frame().round(4)

In [ ]:
import matplotlib.pyplot as plt
f.plot(ci=True)
plt.show()

In [ ]:
from lstm_forecast.ai import generate_insights
print(generate_insights(result, label='AAPL'))  # Claude if ANTHROPIC_API_KEY set, else template